In [ ]:
from fmsae_repro import DATA, MODELS, RESULTS, load_manifest
import os
import joblib
import ezslide
import lazyslide as zs
import mesoslide as ms
import pandas as pd

OLD_CLUSTERIZERS_DIR = f"{MODELS}/token_clusterers/legacy"

In [ ]:
slide = ezslide.read_slide(f'{DATA}/adjacent_registered/LSP24530.zarr', attach_images=False)
slide

In [ ]:
manifest = load_manifest()
slides = ezslide.open_slides(manifest, attach_images=False)

In [ ]:
def plot_correlation(patches, feature):
    import numpy as np
    from scipy.stats import spearmanr
    import matplotlib.pyplot as plt

    cluster_img = patches.tables['tiles_table'].obsm[f'UNI_SAE_{feature}_cluster_img']  # (100, 377, 377)
    obs = patches.tables['tiles_table'].obs

    PIXELS_PER_TOKEN = 725
    n_classes = int(cluster_img.max()) + 1

    token_counts = {}
    for c in range(n_classes):
        counts = (cluster_img == c).sum(axis=(1, 2))
        tokens = np.round(counts / PIXELS_PER_TOKEN).astype(int)
        obs[f'n_tokens_class_{c}'] = tokens
        token_counts[c] = tokens

    colors = plt.cm.viridis(np.linspace(0, 1, n_classes))  # matches cluster map's cmap='viridis'

    fig, ax = plt.subplots(figsize=(6, 5))
    for c in range(n_classes):
        rho, pval = spearmanr(obs[f'_feature_score'], token_counts[c])
        ax.scatter(
            obs[f'_feature_score'], token_counts[c],
            s=24, color=colors[c % len(colors)], alpha=0.8,
            label=f'Class {c} (Spearman ρ={rho:.2f}, p={pval:.2g})',
        )

    ax.set_xlabel(f'_feature_score')
    ax.set_ylabel('Number of tokens')
    ax.set_title('Feature score vs. tokens per cluster class')
    ax.legend(loc='best', fontsize=8, frameon=False)
    fig.tight_layout()
    plt.show()

In [ ]:
OLD_CLUSTERIZERS_DIR = f"{MODELS}/token_clusterers/legacy"
clusterizer_959 = joblib.load(os.path.join(OLD_CLUSTERIZERS_DIR, "UNI_SAE_959.joblib"))
clusterizer_59985 = joblib.load(os.path.join(OLD_CLUSTERIZERS_DIR, "UNI_SAE_59985.joblib"))

In [ ]:
patches_959_all = ms.select_top_patches(slides, f"UNI_SAE_959", n=100, top_fraction=1.0)
patches_959_top = ms.select_top_patches(slides, f"UNI_SAE_959", n=100, top_fraction=0.1)
patches_59985_all = ms.select_top_patches(slides, f"UNI_SAE_59985", n=100, top_fraction=1.0)
patches_59985_top = ms.select_top_patches(slides, f"UNI_SAE_59985", n=100, top_fraction=0.1)

In [ ]:
# 4. Visualize alongside H&E patches.
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches_959_all, clusterizers=[clusterizer_959],
    return_fig=True,
    cache=True,
    cmap='viridis'
)

In [ ]:
plot_correlation(patches_959_all, 959)

In [ ]:
# 4. Visualize alongside H&E patches.
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches_959_top, clusterizers=[clusterizer_959],
    return_fig=True,
    cache=True,
    cmap='viridis'
)

In [ ]:
plot_correlation(patches_959_top, 959)

In [ ]:
# 4. Visualize alongside H&E patches.
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches_59985_all, clusterizers=[clusterizer_59985],
    return_fig=True,
    cache=True,
    cmap='viridis'
)

In [ ]:
plot_correlation(patches_59985_all, 59985)

In [ ]:
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches_59985_top, clusterizers=[clusterizer_59985],
    return_fig=True,
    cache=True,
    cmap='viridis'
)

In [ ]:
plot_correlation(patches_59985_top, 59985)

### Try it on a different histomorphology: MNGC

In [ ]:
patches_56436_full = ms.select_top_patches(slides, f"UNI_SAE_56436", n=100, top_fraction=1.0)
clusterizer_56436 = joblib.load(os.path.join(OLD_CLUSTERIZERS_DIR, "UNI_SAE_56436.joblib"))

_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches_56436_full, clusterizers=[clusterizer_56436],
    return_fig=True,
    cache=True,
    cmap='viridis'
)

In [ ]:
plot_correlation(patches_56436_full, 56436)

### Now test with the updated mesoslide package, the cluster order above should be fixed...

In [ ]:
import torch
clusterizer_56436 = joblib.load(os.path.join(OLD_CLUSTERIZERS_DIR, "UNI_SAE_56436.joblib"))
clusterizer_56436.device = 'cuda:1'

In [ ]:
clusterizer_56436.fit(slides, feature_name='UNI_SAE_56436')

In [ ]:
clusterizer_56436.plot_cluster_feature_correlation()

In [ ]:
patches_56436_full = ms.select_top_patches(slides, f"UNI_SAE_56436", n=20, top_fraction=0.1)

_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches_56436_full, clusterizers=[clusterizer_56436],
    return_fig=True,
    cache=True,
    cmap='viridis'
)

### Compare with a new clusterizer fit on the updated embeddings

In [ ]:
from lazyslide_models import MODEL_REGISTRY

model_module = MODEL_REGISTRY["uni"]
uni = model_module()  # Initiate the model
uni

In [ ]:
from mesoslide.tools.segmenters.TokenClusterizer import TokenClusterizer
clusterizer_56436_new = TokenClusterizer(
    model=uni,
    feature_name='UNI_SAE_56436',
    device='cuda:1'
)

In [ ]:
clusterizer_56436_new.fit(slides, feature_name='UNI_SAE_56436', model=uni)

In [ ]:
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches_56436_full, clusterizers=[clusterizer_56436_new],
    return_fig=True,
    cache=True,
    cmap='viridis'
)

In [ ]:
clusterizer_56436_new.plot_cluster_feature_correlation()

Which looks pretty much the same as before, nice!

### Re-compute cluster order for all 

In [ ]:
from glob import glob
from tqdm import tqdm 
old_clusterizer_paths = glob(os.path.join(OLD_CLUSTERIZERS_DIR, "UNI_SAE_*.joblib"))
old_clusterizer_paths.sort()

for clusterizer_path in tqdm(old_clusterizer_paths):
    feature_name = os.path.basename(clusterizer_path).replace(".joblib", "")
    print(f"Processing {feature_name}...")
    output_path = f'{MODELS}/token_clusterers/updated/{feature_name}.joblib'    
    if os.path.exists(output_path):
        continue    
    clusterizer = joblib.load(clusterizer_path)
    clusterizer.device = 'cuda:1'
    clusterizer.fit(slides, feature_name=feature_name)
    clusterizer.device = 'cpu'
    joblib.dump(clusterizer, output_path)


### Test the updated clusterizers

In [ ]:
from lazyslide_models import MODEL_REGISTRY

model_module = MODEL_REGISTRY["uni"]
uni = model_module()  # Initiate the model

In [ ]:


selected_features = [5196, 56436, 46327, 61987]
clusterizers = []

for feature in selected_features:
    feature_name = f"UNI_SAE_{feature}"
    clusterizer_path = f'{MODELS}/token_clusterers/updated/{feature_name}.joblib'
    clusterizer = joblib.load(clusterizer_path)
    clusterizers.append(clusterizer)
    
patches = ms.select_top_patches(slide, f"UNI_SAE_56436", n=20, top_fraction=0.1)


In [ ]:
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches, clusterizers=clusterizers,
    return_fig=True,
    cache=True,
    cmap='viridis',
    model=uni
)

In [ ]:
# MNGC, EH, TZ, AM
selected_features = [5196, 3207, 55011, 35504] # MNGC, EH1, EH2, EH3
clusterizers = []

for feature in selected_features:
    feature_name = f"UNI_SAE_{feature}"
    clusterizer_path = f'{MODELS}/token_clusterers/updated/{feature_name}.joblib'
    clusterizer = joblib.load(clusterizer_path)
    clusterizers.append(clusterizer)
    
patches = ms.select_top_patches(slide, f"UNI_SAE_56436", n=20, top_fraction=0.1)


In [ ]:
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches, clusterizers=clusterizers,
    return_fig=True,
    cache=True,
    cmap='viridis',
    model=uni
)

In [ ]:
patches_eh = ms.select_top_patches(slide, f"UNI_SAE_3207", n=10, top_fraction=0.1)


In [ ]:
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches_eh, clusterizers=clusterizers,
    return_fig=True,
    cache=True,
    cmap='viridis',
    model=uni
)

In [ ]:
ms.preprocessing.extract_cluster_maps(
    patches, clusterizers[3], model=uni
)

In [ ]:
import zarr
# patches.write(f'{RESULTS}/SAE_feature_similarity/test_patches.zarr')
zarr.consolidate_metadata(f'{RESULTS}/SAE_feature_similarity/test_patches.zarr')